
# Buy Fee Model V11 — Segment Response Models

## Objective

V10 showed that Buy Fee response is highly heterogeneous.

The strongest diagnostic differences came from:

- Location
- Lane
- Price Tier
- Seasonality
- Baseline RPU
- Baseline Volume
- Relative Fee Change %

V11 therefore moves away from a single universal national elasticity.

It focuses on three historically supported pricing scenarios:

- +$10
- +$15
- +$20

and does three things:

1. **Interaction analysis**
   - Location × Price Tier
   - Lane × Price Tier
   - Fee Change % × Previous Buy Fee
   - Fee Change % × Baseline RPU
   - Fee Change % × Baseline Volume
   - Season × Fee Change

2. **Shrinkage for small segments**
   Sparse Location / Lane / Tier results are pulled toward the corporate average rather than treated as equally reliable.

3. **Separate scenario models**
   +$10, +$15, and +$20 are modeled separately.

## Outcomes

For each scenario we model:

- DiD RPU response
- DiD Conversion response
- DiD Volume response

The notebook also reports:

- grouped out-of-time validation by `Corporate_Change_ID`
- MAE
- R²
- direction accuracy
- shrunk segment response
- scenario-specific diagnostics

## Important

+$25 and +$30 are not modeled in V11 because V10 showed too few independent change dates for reliable estimation.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v11_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [10, 15, 20]

RANDOM_STATE = 42

# Scenario matching tolerance
SCENARIO_TOLERANCE = 2.5

# Shrinkage strength
# Higher = more conservative shrinkage toward corporate average
SHRINKAGE_K = 25

# Minimum events for displaying raw segment diagnostics
MIN_SEGMENT_EVENTS = 5

# Validation
MIN_TRAIN_GROUPS = 6
TEST_GROUPS_PER_FOLD = 2

print("=" * 80)
print("V11 INPUT CHECK")
print("=" * 80)

print("DiD file:")
print(DID_FILE)

print("\nExists:")
print(DID_FILE.exists())

if not DID_FILE.exists():
    raise FileNotFoundError(DID_FILE)

print("\nOutput folder:")
print(OUTPUT_DIR)


V11 INPUT CHECK
DiD file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v8_2_outputs\04_DID_EVENT_LEVEL_RESULTS.csv

Exists:
True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v11_outputs


## 1. Load matched-control DiD event data

In [2]:

df = pd.read_csv(
    DID_FILE,
    low_memory=True
)

required = [
    "Segment_Change_ID",
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing)
    )

numeric_cols = [
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

for c in numeric_cols:
    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

df["Change_Date"] = pd.to_datetime(
    df["Corporate_Change_ID"],
    errors="coerce"
)

df["Change_Month"] = df["Change_Date"].dt.month
df["Change_Quarter"] = df["Change_Date"].dt.quarter

def season_from_month(m):
    if pd.isna(m):
        return "Unknown"
    if m in [12, 1, 2]:
        return "Winter"
    if m in [3, 4, 5]:
        return "Spring"
    if m in [6, 7, 8]:
        return "Summer"
    return "Fall"

df["Season"] = df["Change_Month"].apply(
    season_from_month
)

print("Rows:", f"{len(df):,}")
print("Locations:", df["Treated_Location"].nunique())
print("Corporate change dates:", df["Corporate_Change_ID"].nunique())

display(df.head())


Rows: 1,824
Locations: 35
Corporate change dates: 37


,Segment_Change_ID,Corporate_Change_ID,BUYFEE_CHANGE_AMOUNT,PREVIOUS_BUYFEE,FEE_CHANGE_PCT,Treated_Location,Treated_Lane,Treated_Price_Tier,Number_of_Controls,Median_Match_Distance,...,DiD_Fee_Pass_Through,pre_registered,pre_sold,pre_conversion,pre_rpu,pre_avg_saleamt,Change_Date,Change_Month,Change_Quarter,Season
0,ATLANTA_A_1000-2000_2024-12-01,2024-12-01,30.0,250.0,0.120000,ATLANTA,A,1000-2000,3,1.308194,...,0.833803,57,38,0.666667,0.000000,1468.421053,2024-12-01,12,4,Winter
1,ATLANTA_A_10000-11000_2024-12-01,2024-12-01,50.0,460.0,0.108696,ATLANTA,A,10000-11000,3,1.505764,...,0.619878,32,14,0.437500,390.000000,10218.750000,2024-12-01,12,4,Winter
2,ATLANTA_A_2000-3000_2024-12-01,2024-12-01,40.0,325.0,0.123077,ATLANTA,A,2000-3000,3,1.029995,...,0.026193,55,34,0.618182,318.088235,2323.636364,2024-12-01,12,4,Winter
3,ATLANTA_A_3000-4000_2024-12-01,2024-12-01,40.0,350.0,0.114286,ATLANTA,A,3000-4000,3,0.912546,...,0.327910,56,29,0.517857,343.103448,3425.000000,2024-12-01,12,4,Winter
4,ATLANTA_A_4000-5000_2024-12-01,2024-12-01,40.0,365.0,0.109589,ATLANTA,A,4000-5000,3,1.079346,...,1.416928,41,17,0.414634,302.941176,4336.585366,2024-12-01,12,4,Winter


## 2. Create scenario datasets

In [3]:

scenario_data = {}

scenario_summary_rows = []

for fee in SCENARIOS:

    d = df[
        df["BUYFEE_CHANGE_AMOUNT"].between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    ].copy()

    scenario_data[fee] = d

    scenario_summary_rows.append({
        "Fee_Increase": fee,
        "Events": len(d),
        "Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),
        "Median_RPU_Effect": d["DiD_RPU_Change_Dollar"].median(),
        "Median_Conversion_Effect_pp": d["DiD_Conversion_Change_pp"].median(),
        "Median_Volume_Effect_Pct": d["DiD_Volume_Change_Pct"].median(),
    })

scenario_summary = pd.DataFrame(
    scenario_summary_rows
)

scenario_summary.to_csv(
    OUTPUT_DIR / "01_SCENARIO_SAMPLE_SUMMARY.csv",
    index=False
)

display(scenario_summary)


,Fee_Increase,Events,Locations,Corporate_Change_Dates,Median_RPU_Effect,Median_Conversion_Effect_pp,Median_Volume_Effect_Pct
0,10,594,16,24,7.630647,0.627338,-0.003620
1,15,84,14,10,10.481982,1.790793,0.044703
2,20,133,12,10,24.466728,1.658328,-0.025353


## 3. Interaction analysis

In [4]:

# Create explicit interaction terms

df["Location_x_Tier"] = (
    df["Treated_Location"].astype(str)
    + " | "
    + df["Treated_Price_Tier"].astype(str)
)

df["Lane_x_Tier"] = (
    df["Treated_Lane"].astype(str)
    + " | "
    + df["Treated_Price_Tier"].astype(str)
)

df["FeePct_x_PreviousFee"] = (
    df["FEE_CHANGE_PCT"]
    * df["PREVIOUS_BUYFEE"]
)

df["FeePct_x_BaselineRPU"] = (
    df["FEE_CHANGE_PCT"]
    * df["pre_rpu"]
)

df["FeePct_x_BaselineVolume"] = (
    df["FEE_CHANGE_PCT"]
    * df["pre_sold"]
)

df["FeePct_x_BaselineConversion"] = (
    df["FEE_CHANGE_PCT"]
    * df["pre_conversion"]
)

interaction_rows = []

interaction_specs = [
    ("Location_x_Tier", "Location × Price Tier"),
    ("Lane_x_Tier", "Lane × Price Tier"),
    ("Season", "Season"),
]

for col, label in interaction_specs:

    grouped = (
        df.groupby(col, dropna=False)
        .agg(
            Events=("Segment_Change_ID", "count"),
            Median_RPU=("DiD_RPU_Change_Dollar", "median"),
            Median_Conversion=("DiD_Conversion_Change_pp", "median"),
            Median_Volume=("DiD_Volume_Change_Pct", "median"),
        )
        .reset_index()
    )

    grouped = grouped[
        grouped["Events"] >= MIN_SEGMENT_EVENTS
    ]

    if len(grouped) > 1:

        interaction_rows.append({
            "Interaction": label,
            "Groups": len(grouped),
            "RPU_Spread": (
                grouped["Median_RPU"].max()
                - grouped["Median_RPU"].min()
            ),
            "Conversion_Spread": (
                grouped["Median_Conversion"].max()
                - grouped["Median_Conversion"].min()
            ),
            "Volume_Spread": (
                grouped["Median_Volume"].max()
                - grouped["Median_Volume"].min()
            ),
            "Events_Represented": grouped["Events"].sum(),
        })

# Numeric interaction correlations
numeric_interactions = [
    "FeePct_x_PreviousFee",
    "FeePct_x_BaselineRPU",
    "FeePct_x_BaselineVolume",
    "FeePct_x_BaselineConversion",
]

for col in numeric_interactions:

    temp = df[
        [
            col,
            "DiD_RPU_Change_Dollar",
            "DiD_Conversion_Change_pp",
            "DiD_Volume_Change_Pct"
        ]
    ].dropna()

    if len(temp) >= 20:

        interaction_rows.append({
            "Interaction": col,
            "Groups": np.nan,
            "RPU_Spread": temp[col].corr(
                temp["DiD_RPU_Change_Dollar"]
            ),
            "Conversion_Spread": temp[col].corr(
                temp["DiD_Conversion_Change_pp"]
            ),
            "Volume_Spread": temp[col].corr(
                temp["DiD_Volume_Change_Pct"]
            ),
            "Events_Represented": len(temp),
        })

interaction_summary = pd.DataFrame(
    interaction_rows
)

interaction_summary.to_csv(
    OUTPUT_DIR / "02_INTERACTION_DIAGNOSTICS.csv",
    index=False
)

display(interaction_summary)


,Interaction,Groups,RPU_Spread,Conversion_Spread,Volume_Spread,Events_Represented
0,Location × Price Tier,122.0,202.853530,36.098695,0.667917,1127
1,Lane × Price Tier,117.0,95.006355,32.515449,0.880491,922
2,Season,4.0,15.242025,7.785528,0.036272,1824
3,FeePct_x_PreviousFee,NaN,-0.065261,0.035942,0.057019,1824
4,FeePct_x_BaselineRPU,NaN,0.044116,0.027711,0.045352,1824
5,FeePct_x_BaselineVolume,NaN,-0.092515,-0.058787,-0.039850,1824
6,FeePct_x_BaselineConversion,NaN,-0.062362,-0.036214,0.000602,1824


## 4. Shrinkage for small segments

In [5]:

def shrink_segment_effects(
    data,
    group_cols,
    outcome_col,
    shrinkage_k=25
):
    """
    Empirical-Bayes-style shrinkage:

    shrunk_effect =
        weight * segment_mean
        + (1-weight) * global_mean

    weight = n / (n + k)
    """

    global_mean = data[
        outcome_col
    ].mean()

    grouped = (
        data.groupby(
            group_cols,
            dropna=False
        )
        .agg(
            Events=("Segment_Change_ID", "count"),
            Raw_Mean=(outcome_col, "mean"),
            Raw_Median=(outcome_col, "median"),
        )
        .reset_index()
    )

    grouped["Shrinkage_Weight"] = (
        grouped["Events"]
        / (
            grouped["Events"]
            + shrinkage_k
        )
    )

    grouped["Global_Mean"] = global_mean

    grouped["Shrunk_Effect"] = (
        grouped["Shrinkage_Weight"]
        * grouped["Raw_Mean"]
        +
        (
            1
            - grouped["Shrinkage_Weight"]
        )
        * global_mean
    )

    return grouped

shrinkage_outputs = {}

for fee in SCENARIOS:

    d = scenario_data[fee]

    for target_name, outcome_col in [
        ("RPU", "DiD_RPU_Change_Dollar"),
        ("Conversion", "DiD_Conversion_Change_pp"),
        ("Volume", "DiD_Volume_Change_Pct"),
    ]:

        result = shrink_segment_effects(
            d.dropna(subset=[outcome_col]),
            [
                "Treated_Location",
                "Treated_Lane",
                "Treated_Price_Tier"
            ],
            outcome_col,
            shrinkage_k=SHRINKAGE_K
        )

        result["Fee_Increase"] = fee
        result["Target"] = target_name

        key = f"{fee}_{target_name}"
        shrinkage_outputs[key] = result

        result.to_csv(
            OUTPUT_DIR
            / f"03_SHRINKAGE_{fee}_{target_name.upper()}.csv",
            index=False
        )

print("Shrinkage outputs created:", len(shrinkage_outputs))


Shrinkage outputs created: 9


## 5. Build lightweight separate scenario models

In [6]:

def make_ohe():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True
        )

def build_ridge_pipeline(X):

    numeric = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(
            X[c]
        )
    ]

    categorical = [
        c for c in X.columns
        if c not in numeric
    ]

    prep = ColumnTransformer([
        (
            "num",
            Pipeline([
                (
                    "impute",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "scale",
                    StandardScaler()
                )
            ]),
            numeric
        ),
        (
            "cat",
            Pipeline([
                (
                    "impute",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "ohe",
                    make_ohe()
                )
            ]),
            categorical
        )
    ])

    return Pipeline([
        ("prep", prep),
        (
            "model",
            Ridge(
                alpha=20.0
            )
        )
    ])

base_features = [
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",

    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "Season",

    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",

    "FeePct_x_PreviousFee",
    "FeePct_x_BaselineRPU",
    "FeePct_x_BaselineVolume",
    "FeePct_x_BaselineConversion",
]

# Recreate interactions on main df scenario subsets
for fee in SCENARIOS:

    d = df[
        df["BUYFEE_CHANGE_AMOUNT"].between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    ].copy()

    d["FeePct_x_PreviousFee"] = (
        d["FEE_CHANGE_PCT"]
        * d["PREVIOUS_BUYFEE"]
    )

    d["FeePct_x_BaselineRPU"] = (
        d["FEE_CHANGE_PCT"]
        * d["pre_rpu"]
    )

    d["FeePct_x_BaselineVolume"] = (
        d["FEE_CHANGE_PCT"]
        * d["pre_sold"]
    )

    d["FeePct_x_BaselineConversion"] = (
        d["FEE_CHANGE_PCT"]
        * d["pre_conversion"]
    )

    scenario_data[fee] = d

model_rows = []
prediction_parts = []

target_map = {
    "RPU": "DiD_RPU_Change_Dollar",
    "Conversion": "DiD_Conversion_Change_pp",
    "Volume": "DiD_Volume_Change_Pct",
}

for fee in SCENARIOS:

    d = scenario_data[fee].copy()

    features = [
        c for c in base_features
        if c in d.columns
    ]

    groups = (
        d[
            [
                "Corporate_Change_ID",
                "Change_Date"
            ]
        ]
        .drop_duplicates()
        .sort_values("Change_Date")
        ["Corporate_Change_ID"]
        .tolist()
    )

    for target_name, target_col in target_map.items():

        temp = d.dropna(
            subset=[target_col]
        ).copy()

        valid_groups = [
            g for g in groups
            if g in set(
                temp["Corporate_Change_ID"]
            )
        ]

        if len(valid_groups) < (
            MIN_TRAIN_GROUPS
            + TEST_GROUPS_PER_FOLD
        ):

            model_rows.append({
                "Fee_Increase": fee,
                "Target": target_name,
                "Status": "INSUFFICIENT_CHANGE_DATES",
                "Events": len(temp),
                "Corporate_Change_Dates": len(valid_groups),
            })

            continue

        fold_num = 0
        train_end = MIN_TRAIN_GROUPS

        fold_metrics = []

        while (
            train_end
            + TEST_GROUPS_PER_FOLD
            <= len(valid_groups)
        ):

            train_groups = valid_groups[
                :train_end
            ]

            test_groups = valid_groups[
                train_end:
                train_end
                + TEST_GROUPS_PER_FOLD
            ]

            train = temp[
                temp["Corporate_Change_ID"].isin(
                    train_groups
                )
            ]

            test = temp[
                temp["Corporate_Change_ID"].isin(
                    test_groups
                )
            ]

            if len(train) < 30 or len(test) < 5:
                train_end += TEST_GROUPS_PER_FOLD
                continue

            fold_num += 1

            model = build_ridge_pipeline(
                train[features]
            )

            model.fit(
                train[features],
                train[target_col]
            )

            pred = model.predict(
                test[features]
            )

            y = test[target_col].values

            fold_metrics.append({
                "R2": (
                    r2_score(y, pred)
                    if len(y) > 1
                    else np.nan
                ),
                "MAE": mean_absolute_error(
                    y,
                    pred
                ),
                "Direction_Accuracy": np.mean(
                    np.sign(pred)
                    == np.sign(y)
                )
            })

            pp = test[
                [
                    "Segment_Change_ID",
                    "Corporate_Change_ID",
                    "Treated_Location",
                    "Treated_Lane",
                    "Treated_Price_Tier"
                ]
            ].copy()

            pp["Fee_Increase"] = fee
            pp["Target"] = target_name
            pp["Actual"] = y
            pp["Predicted"] = pred
            pp["Fold"] = fold_num

            prediction_parts.append(
                pp
            )

            train_end += TEST_GROUPS_PER_FOLD

        if not fold_metrics:

            model_rows.append({
                "Fee_Increase": fee,
                "Target": target_name,
                "Status": "NO_VALID_FOLDS",
                "Events": len(temp),
                "Corporate_Change_Dates": len(valid_groups),
            })

            continue

        fm = pd.DataFrame(
            fold_metrics
        )

        model_rows.append({
            "Fee_Increase": fee,
            "Target": target_name,
            "Status": "OK",
            "Events": len(temp),
            "Corporate_Change_Dates": len(valid_groups),
            "Folds": len(fm),
            "Median_R2": fm["R2"].median(),
            "Median_MAE": fm["MAE"].median(),
            "Median_Direction_Accuracy": fm[
                "Direction_Accuracy"
            ].median(),
            "Positive_R2_Fold_Share": (
                fm["R2"] > 0
            ).mean(),
        })

model_validation = pd.DataFrame(
    model_rows
)

model_predictions = (
    pd.concat(
        prediction_parts,
        ignore_index=True
    )
    if prediction_parts
    else pd.DataFrame()
)

model_validation.to_csv(
    OUTPUT_DIR / "04_SCENARIO_MODEL_VALIDATION.csv",
    index=False
)

model_predictions.to_csv(
    OUTPUT_DIR / "05_SCENARIO_MODEL_PREDICTIONS.csv",
    index=False
)

display(model_validation)


,Fee_Increase,Target,Status,Events,Corporate_Change_Dates,Folds,Median_R2,Median_MAE,Median_Direction_Accuracy,Positive_R2_Fold_Share
0,10,RPU,OK,594,24,9,-0.411441,31.497519,0.642857,0.000000
1,10,Conversion,OK,594,24,9,-0.152166,10.052327,0.500000,0.111111
2,10,Volume,OK,594,24,9,-0.149302,0.292255,0.446429,0.222222
3,15,RPU,OK,84,10,1,-0.683987,34.239182,0.485714,0.000000
4,15,Conversion,OK,84,10,1,-0.278322,14.760686,0.514286,0.000000
5,15,Volume,OK,84,10,1,-0.172740,0.352905,0.428571,0.000000
6,20,RPU,OK,133,10,2,-0.160923,20.850250,0.777941,0.000000
7,20,Conversion,OK,133,10,2,-0.161948,10.923923,0.510294,0.000000
8,20,Volume,OK,133,10,2,-0.215627,0.304007,0.560294,0.000000


## 6. Scenario-specific segment response tables

In [7]:

segment_tables = []

for fee in SCENARIOS:

    d = scenario_data[fee]

    table = (
        d.groupby(
            [
                "Treated_Location",
                "Treated_Lane",
                "Treated_Price_Tier"
            ],
            dropna=False
        )
        .agg(
            Events=("Segment_Change_ID", "count"),
            Change_Dates=("Corporate_Change_ID", "nunique"),

            Median_RPU=("DiD_RPU_Change_Dollar", "median"),
            Median_Conversion=("DiD_Conversion_Change_pp", "median"),
            Median_Volume=("DiD_Volume_Change_Pct", "median"),

            Median_Previous_BuyFee=("PREVIOUS_BUYFEE", "median"),
            Median_Fee_Change_Pct=("FEE_CHANGE_PCT", "median"),
            Median_Baseline_Conversion=("pre_conversion", "median"),
            Median_Baseline_Volume=("pre_sold", "median"),
            Median_Baseline_RPU=("pre_rpu", "median"),
            Median_Sale_Amount=("pre_avg_saleamt", "median"),
        )
        .reset_index()
    )

    table["Fee_Increase"] = fee

    # Merge shrunk outcomes
    for target_name in [
        "RPU",
        "Conversion",
        "Volume"
    ]:

        shrink = shrinkage_outputs[
            f"{fee}_{target_name}"
        ][
            [
                "Treated_Location",
                "Treated_Lane",
                "Treated_Price_Tier",
                "Shrunk_Effect"
            ]
        ].rename(
            columns={
                "Shrunk_Effect":
                f"Shrunk_{target_name}_Effect"
            }
        )

        table = table.merge(
            shrink,
            on=[
                "Treated_Location",
                "Treated_Lane",
                "Treated_Price_Tier"
            ],
            how="left"
        )

    segment_tables.append(
        table
    )

segment_response = pd.concat(
    segment_tables,
    ignore_index=True
)

segment_response.to_csv(
    OUTPUT_DIR / "06_SEGMENT_RESPONSE_TABLE.csv",
    index=False
)

display(
    segment_response.head(25)
)


,Treated_Location,Treated_Lane,Treated_Price_Tier,Events,Change_Dates,Median_RPU,Median_Conversion,Median_Volume,Median_Previous_BuyFee,Median_Fee_Change_Pct,Median_Baseline_Conversion,Median_Baseline_Volume,Median_Baseline_RPU,Median_Sale_Amount,Fee_Increase,Shrunk_RPU_Effect,Shrunk_Conversion_Effect,Shrunk_Volume_Effect
0,ATLANTA,IL,0-1000,1,1,-40.020588,-1.363501,0.129292,200.0,0.050000,0.569620,45.0,142.555556,605.063291,10,6.965769,0.861923,0.027680
1,ATLANTA,IO,0-1000,1,1,32.550977,6.511097,-0.023973,200.0,0.050000,0.284810,45.0,110.222222,506.962025,10,9.756983,1.164793,0.021785
2,BATONROUGE,1,1000-2000,1,1,-61.805607,-3.886555,0.307822,285.0,0.035088,0.405797,28.0,0.000000,1444.927536,10,6.127884,0.764883,0.034546
3,BATONROUGE,3,1000-2000,1,1,-1.769645,-10.078854,-0.343628,285.0,0.035088,0.746269,50.0,0.000000,1477.582090,10,8.436959,0.526718,0.009491
4,BATONROUGE,5,1000-2000,1,1,-59.135511,3.559691,0.079038,265.0,0.037736,0.632653,62.0,0.000000,1347.908163,10,6.230580,1.051277,0.025747
5,BOSTON,A,0-1000,1,1,-15.355671,-16.378719,-0.118150,200.0,0.050000,0.647887,46.0,142.500000,614.436620,10,7.914420,0.284415,0.018163
6,BOSTON,A,1000-2000,1,1,-30.826676,-6.377994,0.031516,270.0,0.037037,0.517986,72.0,116.250000,1406.115108,10,7.319381,0.669058,0.023919
7,BOSTON,A,10000-11000,1,1,75.044444,-5.869241,0.126984,455.0,0.021978,0.466667,21.0,342.857143,10392.666667,10,11.391347,0.688626,0.027591
8,BOSTON,A,11000-12000,1,1,-5.103317,17.334885,0.472180,460.0,0.021739,0.475000,19.0,431.052632,11346.250000,10,8.308741,1.581092,0.040868
9,BOSTON,A,12000-13000,1,1,-30.962768,1.504999,0.234576,470.0,0.021277,0.585366,24.0,464.791667,12436.585366,10,7.314147,0.972250,0.031729


## 7. Segment confidence bands

In [8]:

def segment_confidence(events, change_dates):
    if events >= 25 and change_dates >= 3:
        return "High"
    if events >= 10 and change_dates >= 2:
        return "Medium"
    return "Low"

segment_response["Segment_Confidence"] = (
    segment_response.apply(
        lambda r: segment_confidence(
            r["Events"],
            r["Change_Dates"]
        ),
        axis=1
    )
)

segment_response["RPU_Response_Class"] = np.select(
    [
        segment_response["Shrunk_RPU_Effect"] >= 10,
        segment_response["Shrunk_RPU_Effect"] > 0,
        segment_response["Shrunk_RPU_Effect"] <= 0,
    ],
    [
        "Strong Positive",
        "Positive",
        "Non-Positive"
    ],
    default="Unknown"
)

segment_response["Conversion_Risk_Class"] = np.select(
    [
        segment_response["Shrunk_Conversion_Effect"] >= -1,
        segment_response["Shrunk_Conversion_Effect"] >= -3,
        segment_response["Shrunk_Conversion_Effect"] < -3,
    ],
    [
        "Low",
        "Moderate",
        "High"
    ],
    default="Unknown"
)

segment_response["Volume_Risk_Class"] = np.select(
    [
        segment_response["Shrunk_Volume_Effect"] >= -0.03,
        segment_response["Shrunk_Volume_Effect"] >= -0.10,
        segment_response["Shrunk_Volume_Effect"] < -0.10,
    ],
    [
        "Low",
        "Moderate",
        "High"
    ],
    default="Unknown"
)

segment_response.to_csv(
    OUTPUT_DIR / "07_SEGMENT_RESPONSE_WITH_CONFIDENCE.csv",
    index=False
)

display(
    segment_response[
        [
            "Fee_Increase",
            "Treated_Location",
            "Treated_Lane",
            "Treated_Price_Tier",
            "Events",
            "Change_Dates",
            "Shrunk_RPU_Effect",
            "Shrunk_Conversion_Effect",
            "Shrunk_Volume_Effect",
            "Segment_Confidence",
            "RPU_Response_Class",
            "Conversion_Risk_Class",
            "Volume_Risk_Class"
        ]
    ].head(30)
)


,Fee_Increase,Treated_Location,Treated_Lane,Treated_Price_Tier,Events,Change_Dates,Shrunk_RPU_Effect,Shrunk_Conversion_Effect,Shrunk_Volume_Effect,Segment_Confidence,RPU_Response_Class,Conversion_Risk_Class,Volume_Risk_Class
0,10,ATLANTA,IL,0-1000,1,1,6.965769,0.861923,0.027680,Low,Positive,Low,Low
1,10,ATLANTA,IO,0-1000,1,1,9.756983,1.164793,0.021785,Low,Positive,Low,Low
2,10,BATONROUGE,1,1000-2000,1,1,6.127884,0.764883,0.034546,Low,Positive,Low,Low
3,10,BATONROUGE,3,1000-2000,1,1,8.436959,0.526718,0.009491,Low,Positive,Low,Low
4,10,BATONROUGE,5,1000-2000,1,1,6.230580,1.051277,0.025747,Low,Positive,Low,Low
5,10,BOSTON,A,0-1000,1,1,7.914420,0.284415,0.018163,Low,Positive,Low,Low
6,10,BOSTON,A,1000-2000,1,1,7.319381,0.669058,0.023919,Low,Positive,Low,Low
7,10,BOSTON,A,10000-11000,1,1,11.391347,0.688626,0.027591,Low,Strong Positive,Low,Low
8,10,BOSTON,A,11000-12000,1,1,8.308741,1.581092,0.040868,Low,Positive,Low,Low
9,10,BOSTON,A,12000-13000,1,1,7.314147,0.972250,0.031729,Low,Positive,Low,Low


## 8. Scenario readiness summary

In [9]:

readiness_rows = []

for fee in SCENARIOS:

    for target in [
        "RPU",
        "Conversion",
        "Volume"
    ]:

        row = model_validation[
            (model_validation["Fee_Increase"].eq(fee))
            &
            (model_validation["Target"].eq(target))
        ]

        if row.empty:
            continue

        r = row.iloc[0]

        status = r["Status"]

        if status != "OK":
            readiness = "RESEARCH ONLY"

        else:

            median_r2 = r.get(
                "Median_R2",
                np.nan
            )

            direction = r.get(
                "Median_Direction_Accuracy",
                np.nan
            )

            if (
                pd.notna(median_r2)
                and median_r2 >= 0.10
                and pd.notna(direction)
                and direction >= 0.60
            ):
                readiness = "GO"

            elif (
                pd.notna(direction)
                and direction >= 0.55
            ):
                readiness = "IMPROVE / RESEARCH"

            else:
                readiness = "RESEARCH ONLY"

        readiness_rows.append({
            "Fee_Increase": fee,
            "Target": target,
            "Readiness": readiness,
            "Median_R2": r.get(
                "Median_R2",
                np.nan
            ),
            "Median_MAE": r.get(
                "Median_MAE",
                np.nan
            ),
            "Direction_Accuracy": r.get(
                "Median_Direction_Accuracy",
                np.nan
            ),
            "Events": r.get(
                "Events",
                np.nan
            ),
            "Corporate_Change_Dates": r.get(
                "Corporate_Change_Dates",
                np.nan
            ),
        })

readiness = pd.DataFrame(
    readiness_rows
)

readiness.to_csv(
    OUTPUT_DIR / "08_V11_READINESS_SUMMARY.csv",
    index=False
)

display(readiness)


,Fee_Increase,Target,Readiness,Median_R2,Median_MAE,Direction_Accuracy,Events,Corporate_Change_Dates
0,10,RPU,IMPROVE / RESEARCH,-0.411441,31.497519,0.642857,594,24
1,10,Conversion,RESEARCH ONLY,-0.152166,10.052327,0.500000,594,24
2,10,Volume,RESEARCH ONLY,-0.149302,0.292255,0.446429,594,24
3,15,RPU,RESEARCH ONLY,-0.683987,34.239182,0.485714,84,10
4,15,Conversion,RESEARCH ONLY,-0.278322,14.760686,0.514286,84,10
5,15,Volume,RESEARCH ONLY,-0.172740,0.352905,0.428571,84,10
6,20,RPU,IMPROVE / RESEARCH,-0.160923,20.850250,0.777941,133,10
7,20,Conversion,RESEARCH ONLY,-0.161948,10.923923,0.510294,133,10
8,20,Volume,IMPROVE / RESEARCH,-0.215627,0.304007,0.560294,133,10


## 9. Final executive summary

In [10]:

summary_rows = []

for fee in SCENARIOS:

    d = scenario_data[fee]

    summary_rows.append({
        "Fee_Increase": fee,

        "Events": len(d),

        "Locations": d[
            "Treated_Location"
        ].nunique(),

        "Corporate_Change_Dates": d[
            "Corporate_Change_ID"
        ].nunique(),

        "Median_RPU": d[
            "DiD_RPU_Change_Dollar"
        ].median(),

        "Median_Conversion": d[
            "DiD_Conversion_Change_pp"
        ].median(),

        "Median_Volume": d[
            "DiD_Volume_Change_Pct"
        ].median(),

        "High_Confidence_Segments": (
            (
                segment_response[
                    "Fee_Increase"
                ].eq(fee)
            )
            &
            (
                segment_response[
                    "Segment_Confidence"
                ].eq("High")
            )
        ).sum(),

        "Medium_Confidence_Segments": (
            (
                segment_response[
                    "Fee_Increase"
                ].eq(fee)
            )
            &
            (
                segment_response[
                    "Segment_Confidence"
                ].eq("Medium")
            )
        ).sum(),
    })

executive = pd.DataFrame(
    summary_rows
)

executive.to_csv(
    OUTPUT_DIR / "09_V11_EXECUTIVE_SUMMARY.csv",
    index=False
)

print("=" * 80)
print("V11 COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

display(executive)


V11 COMPLETE

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v11_outputs


,Fee_Increase,Events,Locations,Corporate_Change_Dates,Median_RPU,Median_Conversion,Median_Volume,High_Confidence_Segments,Medium_Confidence_Segments
0,10,594,16,24,7.630647,0.627338,-0.003620,0,0
1,15,84,14,10,10.481982,1.790793,0.044703,0,0
2,20,133,12,10,24.466728,1.658328,-0.025353,0,0



## Files to share back

Please send:

1. `09_V11_EXECUTIVE_SUMMARY.csv`
2. `08_V11_READINESS_SUMMARY.csv`
3. `04_SCENARIO_MODEL_VALIDATION.csv`
4. `02_INTERACTION_DIAGNOSTICS.csv`
5. `07_SEGMENT_RESPONSE_WITH_CONFIDENCE.csv`
6. `06_SEGMENT_RESPONSE_TABLE.csv`
7. `01_SCENARIO_SAMPLE_SUMMARY.csv`

## Interpretation goal

V11 should tell us:

- whether +$10 / +$15 / +$20 need different models,
- whether interaction terms improve out-of-time stability,
- which Location × Lane × Tier combinations have reliable history,
- and whether segment-level shrinkage gives us a safer foundation for the final pricing tool.
